In [ ]:
import pickle
import plotly.express as px
import scipy.stats as stats
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

from python.statistical_test import PerSeriesTrendTestResultStore

In [ ]:
with open(f"metrics/{cfg.MASTER_FOLDER}/per_series_test_results.pkl", 'rb') as f:
    full_results: dict[str, PerSeriesTrendTestResultStore] = pickle.load(f)

datasets = ['web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

# Summary

In [ ]:
decr_corrs = {}
incr_corrs = {}

decr_corrs_bin = {}
incr_corrs_bin = {}

for m in models:
    decr_corrs[m] = {}
    incr_corrs[m] = {}
    decr_corrs_bin[m] = {}
    incr_corrs_bin[m] = {}
    for d in datasets:
        with open(f"metrics/{cfg.MASTER_FOLDER}/{d}/{m}/biasedness_stats.pkl", "rb") as f:
            biasedness_stats = pickle.load(f)
        biasedness_stats['biased_count'] = biasedness_stats.drop(columns=['all_unbiased']).le(0.05).sum(axis=1)
        results = full_results[f'{d}/{m}']
        increasing_test = results.top.increasing[cfg.TEST].per_series
        increasing_test['incr_significant'] = increasing_test['p_value'] < 0.05
        increasing_test = increasing_test.rename(columns={'p_value': 'incr_p'}).drop(columns=['statistic'])

        decreasing_test = results.top.decreasing[cfg.TEST].per_series
        decreasing_test['decr_significant'] = decreasing_test['p_value'] < 0.05
        decreasing_test = decreasing_test.rename(columns={'p_value': 'decr_p'}).drop(columns=['statistic'])

        correlation_table = pd.concat((increasing_test[['incr_p', 'incr_significant']],
                                    decreasing_test[['decr_p', 'decr_significant']],
                                    biasedness_stats[['biased_count', 'all_unbiased']]), axis=1)

        r_dec, p_dec = stats.pearsonr(correlation_table['biased_count'], correlation_table['decr_p'].fillna(1.0))
        r_inc, p_inc = stats.pearsonr(correlation_table['biased_count'], correlation_table['incr_p'].fillna(1.0))

        decr_corrs[m][d] = r_dec if p_dec < 0.05 else "No"
        incr_corrs[m][d] = r_inc if p_inc < 0.05 else "No"

        r_dec, p_dec = stats.pearsonr((correlation_table['biased_count'] <= 1).astype(int), correlation_table['decr_significant'].fillna(False).astype(int))
        r_inc, p_inc = stats.pearsonr((correlation_table['biased_count'] <= 1).astype(int), correlation_table['incr_significant'].fillna(False).astype(int))

        decr_corrs_bin[m][d] = r_dec if p_dec < 0.05 else "No"
        incr_corrs_bin[m][d] = r_inc if p_inc < 0.05 else "No"


In [ ]:
print('Decreasing trend test p_value correlation with biased forecast count')
print('Expecting: positive correlation')
pd.DataFrame(decr_corrs)

In [ ]:
print('Increasing trend test p_value correlation with biased forecast count')
print('Expecting: negative correlation')
pd.DataFrame(incr_corrs)

In [ ]:
print('Signficiant increasing trend correlation with indicator of at most 1 biased base forecast')
print('Expecting: negative correlation')
pd.DataFrame(incr_corrs_bin)

In [ ]:
print('Signficiant decreasing trend correlation with indicator of at most 1 biased base forecast')
print('Expecting: positive correlation')
pd.DataFrame(decr_corrs_bin)

In [ ]:
biasedness_stats

In [ ]:
correlation_table

# Single

In [ ]:
with open(f"metrics/{cfg.MASTER_FOLDER}/per_series_test_results.pkl", 'rb') as f:
    full_results: dict[str, PerSeriesTrendTestResultStore] = pickle.load(f)

with open(f"metrics/{cfg.FOLDER}/biasedness_stats.pkl", "rb") as f:
    biasedness_stats = pickle.load(f)
    biasedness_stats['biased_count'] = biasedness_stats.drop(columns=['all_unbiased']).le(0.05).sum(axis=1)

In [ ]:
results = full_results[f'{cfg.DATASET.NAME}/{cfg.MODEL}']
increasing_test = results.top.increasing[cfg.TEST].per_series
increasing_test['incr_significant'] = increasing_test['p_value'] < 0.05
increasing_test = increasing_test.rename(columns={'p_value': 'incr_p'}).drop(columns=['statistic'])

decreasing_test = results.top.decreasing[cfg.TEST].per_series
decreasing_test['decr_significant'] = decreasing_test['p_value'] < 0.05
decreasing_test = decreasing_test.rename(columns={'p_value': 'decr_p'}).drop(columns=['statistic'])

correlation_table = pd.concat((increasing_test[['incr_p', 'incr_significant']],
                               decreasing_test[['decr_p', 'decr_significant']],
                               biasedness_stats[['all_unbiased', 'biased_count']]), axis=1)

In [ ]:
print('Expecting: High at (true, false); (false, true)')

px.density_heatmap(correlation_table, x='all_unbiased', y='incr_significant',
                   labels={'y': 'significantly_increasing', 'x': 'all_base_forecasts_unbiased'}
).update_xaxes(categoryorder="category ascending"
).update_yaxes(categoryorder="category ascending")

In [ ]:
print('Expecting: High at (true, true); (false, false)')

px.density_heatmap(correlation_table, x='all_unbiased', y='decr_significant',
                   labels={'y': 'significantly_decreasing', 'x': 'all_base_forecasts_unbiased'},
).update_xaxes(categoryorder="category ascending"
).update_yaxes(categoryorder="category ascending")

In [ ]:
px.density_heatmap(correlation_table, x='biased_count', y='incr_p',
                   labels={'y': 'increasing_p_val', 'x': 'all_base_forecasts_unbiased'})

In [ ]:
px.density_heatmap(correlation_table, x='biased_count', y='decr_p',
                   labels={'y': 'decreasing_p_val', 'x': 'all_base_forecasts_unbiased'},
)

In [ ]:
r_dec, p_dec = stats.pearsonr(correlation_table['biased_count'], correlation_table['decr_p'].fillna(1.0))
r_inc, p_inc = stats.pearsonr(correlation_table['biased_count'], correlation_table['incr_p'].fillna(1.0))

print(f"Biasedness count correlation with p_value of decreasing trend: {r_dec:.4f} ; {"SIGNIFICANT" if p_dec < 0.05 else "NOT SIGNIFICANT"} p_value={p_dec}")
print(f"Biasedness count correlation with p_value of increasing trend: {r_inc:.4f} ; {"SIGNIFICANT" if p_inc < 0.05 else "NOT SIGNIFICANT"} p_value={p_inc}")


In [ ]:
results = full_results[f'{cfg.DATASET.NAME}/{cfg.MODEL}']
increasing_test = results.top.increasing[cfg.TEST].per_series
increasing_test['incr_significant'] = increasing_test['p_value'] < 0.05
increasing_test = increasing_test.rename(columns={'p_value': 'incr_p'}).drop(columns=['statistic'])

decreasing_test = results.top.decreasing[cfg.TEST].per_series
decreasing_test['decr_significant'] = decreasing_test['p_value'] < 0.05
decreasing_test = decreasing_test.rename(columns={'p_value': 'decr_p'}).drop(columns=['statistic'])

correlation_table = pd.concat((increasing_test[['incr_p']],
                               decreasing_test[['decr_p']],
                               biasedness_stats[['Top0']]), axis=1)

In [ ]:
correlation_table

In [ ]:
px.scatter(correlation_table, x='decr_p', y='Top0', log_x=False)